# 0.13 — Generative AI: graph-based emergence detection

**Question:** can a temporal **PPMI co-occurrence graph** discover the emerging generative-AI lexical community around Nov–Dec 2022 — with **no keyword filter, no lexicon, no embedding seed phrases** at any stage?

Implements Section 24 of [`alternative_plan_with_graph.md`](alternative_plan_with_graph.md) as a parallel benchmark to BERTrend (`0.9`–`0.12`).

**Pipeline:** phrase extraction → weekly PPMI graphs → Louvain communities → term/edge burst vs baseline → Jaccard tracking → emergence ranking → manual interpretation.

**Milestones (annotation only):** ChatGPT 2022-11-30 · CHAT ETF 2023-05-17

**Outputs:** `genai_graph_terms.parquet`, `genai_graph_communities.parquet`, `genai_graph_emergence_rankings.parquet`, `genai_graph_edge_bursts.parquet`, `genai_graph_validation.parquet`, HTML visualizations

In [ ]:
import json
import lzma
import re
from collections import Counter, defaultdict
from itertools import combinations
from math import log
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import polars as pl
from networkx.algorithms.community import louvain_communities
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from tqdm.auto import tqdm

_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = _ROOT / "data" / "raw"
OUTPUT_DIR = _ROOT / "notebooks" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- window ---
YEARS = [2021, 2022, 2023]
DATE_START = pd.Timestamp("2021-01-01")
DATE_END = pd.Timestamp("2023-12-31")
BASELINE_END = pd.Timestamp("2022-10-31")
DISCOVERY_START = pd.Timestamp("2022-11-01")
DISCOVERY_END = pd.Timestamp("2022-12-31")
CHATGPT_LAUNCH = pd.Timestamp("2022-11-30")
INCEPTION = pd.Timestamp("2023-05-17")
BLOOMBERG_WIRES = ["BN", "BFW", "BBO"]
META_CACHE = OUTPUT_DIR / "genai_full_meta.parquet"
RANDOM_SEED = 42

# --- graph pipeline config ---
FREQ = "W-MON"
MIN_DOC_FREQ = 5
WEEKLY_MIN_COUNT = 2          # per-week graph node floor (lower than corpus df)
MAX_DOC_FREQ_PCT = 0.15           # tighter — drop ubiquitous finance tokens
MIN_WEEK_HEADLINES = 500
MIN_PPMI = 0.0
LOUVAIN_RESOLUTION = 1.0
LOUVAIN_SEED = 42
ALPHA = 1.0
MIN_COMMUNITY_SIZE = 3
JACCARD_LINK = 0.25
TOP_TERMS = 12
TOP_HEADLINES = 10
TOP_RANK_PRINT = 15

TERMS_CACHE = OUTPUT_DIR / "genai_graph_terms.parquet"
COMMUNITIES_PATH = OUTPUT_DIR / "genai_graph_communities.parquet"
RANKINGS_PATH = OUTPUT_DIR / "genai_graph_emergence_rankings.parquet"
EDGE_BURSTS_PATH = OUTPUT_DIR / "genai_graph_edge_bursts.parquet"
VALIDATION_PATH = OUTPUT_DIR / "genai_graph_validation.parquet"

FINANCE_STOP = set(ENGLISH_STOP_WORDS) | {
    "inc", "plc", "ltd", "llc", "corp", "co", "sa", "ag", "nv", "group", "holdings",
    "ceo", "cfo", "says", "said", "new", "year", "today", "week", "day", "update",
    "report", "reports", "results", "announces", "announced", "shares", "stock", "stocks",
    "stake", "dividend", "q1", "q2", "q3", "q4", "fy", "unit", "mln", "bln", "pct",
    "jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec",
    "sales", "deal", "chief", "cut", "raised", "raise", "buy", "sell", "net", "revenue",
    "beat", "miss", "forecast", "outlook", "business", "market", "markets", "global",
    "first", "second", "third", "fourth", "annual", "meeting", "plans", "plan", "bn",
    "march", "april", "june", "july", "august", "september", "october", "november", "december",
    # analyst / ratings boilerplate (dominated BERTrend topics in 0.9)
    "rated", "buy", "sell", "hold", "neutral", "perform", "outperform", "underperform",
    "overweight", "underweight", "equal-weight", "cut", "raise", "raised", "lowers", "upgrade",
    "downgrade", "maintains", "reiterates", "est", "eps", "adj", "sees", "expects", "forecast",
    "names", "appoints", "hires", "officer", "director", "chairman", "executive", "promotes",
    "tender", "offering", "offer", "notes", "bond", "bonds", "debt", "bills", "yield",
}
CUSTOM_STOP = {
    "inc", "plc", "ltd", "llc", "corp", "co", "sa", "ag", "nv", "group", "holdings",
    "ceo", "cfo", "says", "said", "new", "year", "today", "week", "day", "update",
    "report", "reports", "results", "announces", "announced", "shares", "stock",
    "stocks", "stake", "dividend", "q1", "q2", "q3", "q4", "fy", "unit", "mln", "bln", "pct",
}
TERM_STOP = FINANCE_STOP | CUSTOM_STOP
TOKEN_RE = re.compile(r"(?u)\b[a-z][a-z0-9\-]{2,}\b")
BIGRAM_RE = re.compile(r"(?u)\b[a-z][a-z0-9\-]{2,}\s+[a-z][a-z0-9\-]{2,}\b")

print(f"Window {DATE_START.date()}→{DATE_END.date()} | baseline through {BASELINE_END.date()}")

## 1–2. Load headlines + phrase extraction

Primary load: `genai_full_meta.parquet` from [`0.10`](0.10-genai-hierarchy-drilldown.ipynb). Extract filtered unigrams + bigrams per headline; cache to `genai_graph_terms.parquet`.

In [ ]:
def strip_prefix(text: str) -> str:
    for _ in range(2):
        if ":" not in text:
            return text
        prefix, _, rest = text.partition(":")
        if not prefix or not rest or len(prefix) > 30 or len(prefix.split()) > 4:
            return text
        text = rest.strip()
    return text


def extract_headline_terms(text: str) -> list[str]:
    t = text.lower()
    terms = set()
    for bg in BIGRAM_RE.findall(t):
        toks = bg.split()
        if all(tok not in TERM_STOP for tok in toks):
            terms.add(bg)
    for tok in TOKEN_RE.findall(t):
        if tok not in TERM_STOP and len(tok) >= 3:
            terms.add(tok)
    return sorted(terms)


def load_headlines() -> pd.DataFrame:
    if META_CACHE.exists():
        df = pd.read_parquet(META_CACHE)
        df["date"] = pd.to_datetime(df["date"])
        df = df[(df.date >= DATE_START) & (df.date <= DATE_END)].reset_index(drop=True)
        print(f"Loaded {META_CACHE.name}: {len(df):,} headlines")
        return df
    frames = []
    for yr in YEARS:
        path = RAW_DIR / f"raw_news_{yr}.csv.xz"
        with lzma.open(path, "rb") as f:
            part = (
                pl.scan_csv(f, infer_schema_length=10_000)
                .select(["Headline", "CaptureTime", "WireName"])
                .filter(pl.col("WireName").is_in(BLOOMBERG_WIRES) & pl.col("Headline").is_not_null())
                .with_columns(pl.col("CaptureTime").str.to_datetime(time_zone="UTC", strict=False))
                .collect()
            )
        frames.append(part)
        print(f"  {yr}: {part.height:>9,} Bloomberg rows")
    df = pl.concat(frames).to_pandas()
    df["date"] = pd.to_datetime(df["CaptureTime"]).dt.tz_localize(None)
    df = df[(df.date >= DATE_START) & (df.date <= DATE_END)]
    df = df.dropna(subset=["Headline"]).drop_duplicates("Headline")
    df["Headline"] = df["Headline"].map(strip_prefix)
    df = df[df.Headline.str.split().map(len) >= 4].reset_index(drop=True)
    print(f"Total headlines: {len(df):,}")
    return df


if TERMS_CACHE.exists():
    news = pd.read_parquet(TERMS_CACHE)
    news["date"] = pd.to_datetime(news["date"])
    news["week"] = news["week"].astype(str)
    print(f"Loaded cache {TERMS_CACHE.name}: {len(news):,} rows")
else:
    news = load_headlines()
    news["week"] = news["date"].dt.to_period(FREQ).astype(str)
    news["hl_lc"] = news["Headline"].str.lower()
    terms_list = []
    for text in tqdm(news["Headline"], desc="extract terms"):
        terms_list.append(extract_headline_terms(text))
    news["terms"] = terms_list
    news.to_parquet(TERMS_CACHE, index=False)
    print(f"Wrote {TERMS_CACHE.name}: {len(news):,} rows")

if "hl_lc" not in news.columns:
    news["hl_lc"] = news["Headline"].str.lower()
print(f"Headlines with ≥1 term: {(news['terms'].map(len) > 0).sum():,}")

## 3–5. Corpus filter, weekly PPMI graphs, Louvain communities

In [ ]:
def build_vocab(doc_terms: pd.Series, n_docs: int) -> set[str]:
    dfreq = Counter()
    for terms in doc_terms:
        for t in set(terms):
            dfreq[t] += 1
    max_df = int(MAX_DOC_FREQ_PCT * n_docs)
    vocab = {t for t, c in dfreq.items() if MIN_DOC_FREQ <= c <= max_df}
    print(f"Vocab: {len(vocab):,} terms (of {len(dfreq):,} raw; df {MIN_DOC_FREQ}–{max_df})")
    return vocab


def filter_terms(terms: list[str], vocab: set[str]) -> list[str]:
    out = []
    for t in terms:
        if t not in vocab:
            continue
        if any(tok in TERM_STOP for tok in t.split()):
            continue
        out.append(t)
    return sorted(out)


def build_ppmi_graph(rows: list[list[str]], min_count: int = MIN_DOC_FREQ) -> nx.Graph:
    term_counts = Counter()
    pair_counts = Counter()
    n_documents = len(rows)
    if n_documents == 0:
        return nx.Graph()
    for phrases in rows:
        unique_terms = set(phrases)
        term_counts.update(unique_terms)
        pair_counts.update(combinations(sorted(unique_terms), 2))
    graph = nx.Graph()
    for term, count in term_counts.items():
        if count >= min_count:
            graph.add_node(term, count=count)
    for (term_a, term_b), pair_count in pair_counts.items():
        if term_a not in graph or term_b not in graph:
            continue
        p_ab = pair_count / n_documents
        p_a = term_counts[term_a] / n_documents
        p_b = term_counts[term_b] / n_documents
        ppmi = max(0.0, log(p_ab / (p_a * p_b)) if p_a * p_b > 0 else 0.0)
        if ppmi > MIN_PPMI:
            graph.add_edge(term_a, term_b, weight=ppmi, count=pair_count)
    return graph


def weighted_degree(graph: nx.Graph, node: str) -> float:
    return sum(graph[node][nbr].get("weight", 0.0) for nbr in graph.neighbors(node))


def community_coherence(graph: nx.Graph, terms: set[str]) -> float:
    if len(terms) < 2:
        return 0.0
    weights = []
    for a, b in combinations(sorted(terms), 2):
        if graph.has_edge(a, b):
            weights.append(graph[a][b].get("weight", 0.0))
        else:
            weights.append(0.0)
    return float(np.mean(weights)) if weights else 0.0


def detect_communities(graph: nx.Graph) -> list[dict]:
    if graph.number_of_nodes() == 0:
        return []
    comms = louvain_communities(
        graph, weight="weight", resolution=LOUVAIN_RESOLUTION, seed=LOUVAIN_SEED,
    )
    out = []
    for idx, term_set in enumerate(comms):
        if len(term_set) < MIN_COMMUNITY_SIZE:
            continue
        terms = set(term_set)
        ranked = sorted(terms, key=lambda t: weighted_degree(graph, t), reverse=True)
        internal_edges = []
        for a, b in combinations(sorted(terms), 2):
            if graph.has_edge(a, b):
                internal_edges.append((a, b, graph[a][b].get("count", 0)))
        out.append({
            "community_id": idx,
            "terms": terms,
            "top_terms": ranked[:TOP_TERMS],
            "coherence": community_coherence(graph, terms),
            "internal_edges": internal_edges,
            "n_terms": len(terms),
        })
    return out


vocab = build_vocab(news["terms"], len(news))
news["terms_f"] = news["terms"].map(lambda ts: filter_terms(ts, vocab))

weekly_rows = []
skipped_weeks = []
graphs_by_week = {}
communities_by_week = {}

for week, grp in tqdm(news.groupby("week", sort=True), desc="weekly graphs"):
    rows = grp["terms_f"].tolist()
    if len(rows) < MIN_WEEK_HEADLINES:
        skipped_weeks.append((week, len(rows), "too_few_headlines"))
        continue
    graph = build_ppmi_graph(rows, min_count=WEEKLY_MIN_COUNT)
    if graph.number_of_nodes() < MIN_COMMUNITY_SIZE:
        skipped_weeks.append((week, len(rows), "too_few_nodes"))
        continue
    graphs_by_week[week] = graph
    comms = detect_communities(graph)
    communities_by_week[week] = comms
    for c in comms:
        weekly_rows.append({
            "period": week,
            "community_id": c["community_id"],
            "top_terms": ", ".join(c["top_terms"]),
            "n_terms": c["n_terms"],
            "coherence": c["coherence"],
            "n_internal_edges": len(c["internal_edges"]),
        })

print(f"Weeks with graphs: {len(graphs_by_week)} | skipped: {len(skipped_weeks)}")
if skipped_weeks[:5]:
    print("Sample skipped:", skipped_weeks[:5])

## 6–7. Term/edge burst vs baseline + community tracking

In [ ]:
def jaccard(a: set[str], b: set[str]) -> float:
    if not a and not b:
        return 0.0
    return len(a & b) / len(a | b)


def week_ts(week_str: str) -> pd.Timestamp:
    return pd.Period(week_str, freq=FREQ).start_time


baseline_weeks = [w for w in graphs_by_week if week_ts(w) <= BASELINE_END]
print(f"Baseline weeks: {len(baseline_weeks)} | all weeks: {len(graphs_by_week)}")

# --- per-week term and edge counts ---
term_week_counts = defaultdict(lambda: defaultdict(int))
edge_week_counts = defaultdict(lambda: defaultdict(int))
week_headline_counts = {}

for week, grp in news.groupby("week"):
    week_headline_counts[week] = len(grp)

for week, graph in graphs_by_week.items():
    rows = news.loc[news.week == week, "terms_f"].tolist()
    for terms in rows:
        for t in set(terms):
            term_week_counts[t][week] += 1
    for a, b in graph.edges():
        edge_week_counts[(a, b)][week] += graph[a][b].get("count", 0)


def baseline_expectation(counter_by_week: dict, baseline_ws: list[str]) -> float:
    if not baseline_ws:
        return 0.0
    return float(np.mean([counter_by_week.get(w, 0) for w in baseline_ws]))


def term_burst(term: str, week: str) -> float:
    obs = term_week_counts[term].get(week, 0)
    exp = baseline_expectation(term_week_counts[term], baseline_weeks)
    return log((obs + ALPHA) / (exp + ALPHA))


def edge_burst(a: str, b: str, week: str) -> float:
    key = (a, b) if a < b else (b, a)
    obs = edge_week_counts[key].get(week, 0)
    exp = baseline_expectation(edge_week_counts[key], baseline_weeks)
    return log((obs + ALPHA) / (exp + ALPHA))


def baseline_term_presence(term: str) -> float:
    return baseline_expectation(term_week_counts[term], baseline_weeks)


def baseline_edge_presence(a: str, b: str) -> float:
    key = (a, b) if a < b else (b, a)
    return baseline_expectation(edge_week_counts[key], baseline_weeks)


# --- assign track ids via Jaccard matching ---
next_track = 0
prev_comms = []
track_records = []
headline_membership = defaultdict(list)  # (week, track_id) -> headline indices

for week in sorted(communities_by_week.keys(), key=week_ts):
    grp = news.loc[news.week == week]
    hl_idx = grp.index.tolist()
    hl_terms = grp["terms_f"].tolist()
    matched_prev = set()
    week_comms = []
    for c in communities_by_week[week]:
        best_j, best_track = 0.0, None
        for pc in prev_comms:
            if pc["track_id"] in matched_prev:
                continue
            j = jaccard(c["terms"], pc["terms"])
            if j >= JACCARD_LINK and j > best_j:
                best_j, best_track = j, pc["track_id"]
        if best_track is None:
            best_track = next_track
            next_track += 1
        else:
            matched_prev.add(best_track)
        c = dict(c)
        c["track_id"] = best_track
        c["period"] = week
        # headline count + membership
        members = []
        for i, terms in zip(hl_idx, hl_terms):
            overlap = c["terms"] & set(terms)
            if overlap:
                members.append(i)
        c["headline_count"] = len(members)
        c["headline_indices"] = members
        headline_membership[(week, best_track)] = members
        graph = graphs_by_week[week]
        c["top_terms"] = sorted(
            c["terms"],
            key=lambda t: weighted_degree(graph, t) * np.exp(term_burst(t, week)),
            reverse=True,
        )[:TOP_TERMS]
        # community-level burst/novelty
        tb = [term_burst(t, week) for t in c["terms"]]
        eb = [edge_burst(a, b, week) for a, b, _ in c["internal_edges"]]
        c["mean_term_burst"] = float(np.mean(tb)) if tb else 0.0
        c["mean_edge_burst"] = float(np.mean(eb)) if eb else 0.0
        c["max_edge_burst"] = float(max(eb)) if eb else 0.0
        c["vocabulary_novelty"] = float(np.mean([
            1.0 if baseline_term_presence(t) < 0.5 else 0.0 for t in c["terms"]
        ]))
        c["edge_novelty"] = float(np.mean([
            1.0 if baseline_edge_presence(a, b) < 0.5 else 0.0
            for a, b, _ in c["internal_edges"]
        ])) if c["internal_edges"] else 0.0
        week_comms.append(c)
        track_records.append(c)
    prev_comms = week_comms

baseline_hc = float(np.mean([
    c["headline_count"] for c in track_records if c["period"] in baseline_weeks
])) if baseline_weeks else 1.0
for c in track_records:
    c["frequency_growth"] = log((c["headline_count"] + ALPHA) / (baseline_hc + ALPHA))

# persistence per track
track_weeks = defaultdict(list)
for c in track_records:
    track_weeks[c["track_id"]].append(c["period"])
for c in track_records:
    ws = sorted(track_weeks[c["track_id"]], key=week_ts)
    streak = 1
    for i in range(1, len(ws)):
        if (week_ts(ws[i]) - week_ts(ws[i - 1])).days <= 8:
            streak += 1
        else:
            break
    c["persistence"] = streak
    c["first_seen"] = ws[0]

print(f"Community-week records: {len(track_records):,} | tracks: {next_track:,}")

## 8–9. Emergence ranking + representative headlines

In [ ]:
FEATURES = [
    "frequency_growth", "mean_term_burst", "mean_edge_burst",
    "vocabulary_novelty", "edge_novelty", "coherence", "persistence",
]


def zscore_series(s: pd.Series) -> pd.Series:
    if s.std(ddof=0) == 0 or len(s) < 2:
        return pd.Series(0.0, index=s.index)
    return (s - s.mean()) / s.std(ddof=0)


rows = []
for c in track_records:
    rows.append({
        "period": c["period"],
        "community_id": c["community_id"],
        "track_id": c["track_id"],
        "top_terms": ", ".join(c["top_terms"]),
        "headline_count": c["headline_count"],
        "first_seen": c.get("first_seen", c["period"]),
        **{f: c[f] for f in FEATURES},
    })
rank_df = pd.DataFrame(rows)

rank_df["emergence_score"] = 0.0
for week, sub in rank_df.groupby("period"):
    zcols = [zscore_series(sub[f]) for f in FEATURES]
    rank_df.loc[sub.index, "emergence_score"] = np.mean(np.vstack(zcols), axis=0)

rank_df["rank_in_week"] = rank_df.groupby("period")["emergence_score"].rank(ascending=False, method="first").astype(int)
rank_df = rank_df.sort_values(["period", "rank_in_week"]).reset_index(drop=True)
rank_df.to_parquet(RANKINGS_PATH, index=False)
print(f"Wrote {RANKINGS_PATH.name}: {len(rank_df):,} rows")

# representative headlines for top communities in discovery window
comm_lookup = {(c["period"], c["track_id"]): c for c in track_records}
rep_rows = []
disc_weeks = rank_df.loc[
    (rank_df["period"].map(week_ts) >= DISCOVERY_START)
    & (rank_df["period"].map(week_ts) <= DISCOVERY_END)
]
for _, r in disc_weeks[disc_weeks["rank_in_week"] <= TOP_RANK_PRINT].iterrows():
    c = comm_lookup.get((r["period"], r["track_id"]))
    if c is None:
        continue
    graph = graphs_by_week.get(r["period"])
    term_w = {t: weighted_degree(graph, t) * np.exp(term_burst(t, r["period"])) for t in c["terms"]} if graph else {}
    scores = []
    for idx in c["headline_indices"][:5000]:
        hl = news.at[idx, "Headline"]
        terms = set(news.at[idx, "terms_f"])
        score = sum(term_w.get(t, 0.0) for t in terms & c["terms"])
        scores.append((score, hl))
    scores.sort(key=lambda x: -x[0])
    for rank, (score, hl) in enumerate(scores[:TOP_HEADLINES], 1):
        rep_rows.append({
            "period": r["period"], "track_id": r["track_id"], "rank_in_week": int(r["rank_in_week"]),
            "rep_rank": rank, "score": score, "headline": hl[:240],
        })

rep_df = pd.DataFrame(rep_rows)

# communities parquet
comm_df = pd.DataFrame([{
    "period": c["period"], "community_id": c["community_id"], "track_id": c["track_id"],
    "top_terms": ", ".join(c["top_terms"]), "headline_count": c["headline_count"],
    "coherence": c["coherence"], "mean_term_burst": c["mean_term_burst"],
    "mean_edge_burst": c["mean_edge_burst"], "vocabulary_novelty": c["vocabulary_novelty"],
    "edge_novelty": c["edge_novelty"], "persistence": c["persistence"],
    "first_seen": c.get("first_seen", c["period"]),
} for c in track_records])
comm_df.to_parquet(COMMUNITIES_PATH, index=False)
print(f"Wrote {COMMUNITIES_PATH.name}")

print("\nTop emerging communities — discovery window (Nov–Dec 2022):")
for week in sorted(disc_weeks["period"].unique(), key=week_ts):
    sub = rank_df[(rank_df.period == week)].head(5)
    print(f"\n  {week}")
    for _, r in sub.iterrows():
        print(f"    #{int(r.rank_in_week)} track {int(r.track_id)} score {r.emergence_score:.2f}  {r.top_terms[:70]}")

## 10–12. Post-hoc validation, edge bursts, visualizations

In [ ]:
# --- post-hoc genAI lexicon (validation only) ---
GENAI_T1_STRICT = [
    r"generative ai", r"generative artificial intelligence",
    r"large language model", r"large language models", r"\bllms\b",
    r"chatgpt", r"gpt-3\.5", r"gpt-3", r"gpt-4",
    r"foundation model", r"foundation models",
    r"stable diffusion", r"midjourney", r"dall-e", r"dalle", r"text-to-image", r"text to image",
]
GENAI_T2_STANDARD = [
    r"\bopenai\b", r"\banthropic\b",
    r"chatgpt-like", r"chatgpt-style", r"prompt engineering",
    r"ai chatbot", r"ai chat bot",
]
GENAI_STANDARD = re.compile(
    "|".join(f"(?:{p})" for p in GENAI_T1_STRICT + GENAI_T2_STANDARD), re.I,
)


def pct_genai(headlines: list[str]) -> float:
    if not headlines:
        return 0.0
    s = pd.Series(headlines)
    return round(s.str.contains(GENAI_STANDARD, na=False).mean() * 100, 1)


val_rows = []
for c in track_records:
    wts = week_ts(c["period"])
    if wts < DISCOVERY_START or wts > DISCOVERY_END:
        continue
    hls = [news.at[i, "Headline"] for i in c["headline_indices"]]
    rep_hls = rep_df.loc[
        (rep_df.period == c["period"]) & (rep_df.track_id == c["track_id"]), "headline"
    ].tolist() if len(rep_df) else []
    val_rows.append({
        "period": c["period"], "track_id": c["track_id"],
        "rank_in_week": int(rank_df.loc[
            (rank_df.period == c["period"]) & (rank_df.track_id == c["track_id"]), "rank_in_week"
        ].iloc[0]),
        "emergence_score": float(rank_df.loc[
            (rank_df.period == c["period"]) & (rank_df.track_id == c["track_id"]), "emergence_score"
        ].iloc[0]),
        "pct_standard_all": pct_genai(hls),
        "pct_standard_rep": pct_genai(rep_hls),
        "top_terms": ", ".join(c["top_terms"]),
        "first_seen": c.get("first_seen", c["period"]),
    })
val_df = pd.DataFrame(val_rows).sort_values(["period", "rank_in_week"])
val_df.to_parquet(VALIDATION_PATH, index=False)
print(f"Wrote {VALIDATION_PATH.name}")

best = val_df.sort_values(["pct_standard_rep", "pct_standard_all", "emergence_score"], ascending=False).head(10)
print("\nBest genAI overlap (post-hoc STANDARD lexicon):")
print(best[["period", "rank_in_week", "pct_standard_rep", "pct_standard_all", "top_terms"]].to_string(index=False))

bertrend_baseline = 1.2  # 0.12 all-news best_pct_standard
if len(best) and best.iloc[0]["pct_standard_rep"] >= 20:
    print(f"\n=> PASS: rep-headline overlap {best.iloc[0]['pct_standard_rep']}% (≥20%; BERTrend baseline {bertrend_baseline}%)")
else:
    top_pct = best.iloc[0]["pct_standard_rep"] if len(best) else 0
    print(f"\n=> Review: best rep overlap {top_pct}% (target ≥20%; BERTrend baseline {bertrend_baseline}%)")

# --- diagnostic: communities whose *member terms* include genAI vocabulary ---
GENAI_TERM_HINT = re.compile(
    r"chatgpt|openai|chatbot|generative|gpt-3|gpt-4|anthropic|copilot|bard|gemini|llm", re.I,
)
diag_rows = []
for c in track_records:
    wts = week_ts(c["period"])
    if wts < DISCOVERY_START or wts > DISCOVERY_END:
        continue
    hit_terms = [t for t in c["terms"] if GENAI_TERM_HINT.search(t)]
    if not hit_terms:
        continue
    hls = [news.at[i, "Headline"] for i in c["headline_indices"]]
    rk = rank_df.loc[(rank_df.period == c["period"]) & (rank_df.track_id == c["track_id"])]
    diag_rows.append({
        "period": c["period"], "track_id": c["track_id"],
        "rank_in_week": int(rk.rank_in_week.iloc[0]) if len(rk) else None,
        "emergence_score": float(rk.emergence_score.iloc[0]) if len(rk) else None,
        "genai_terms": ", ".join(sorted(hit_terms)[:8]),
        "top_terms": ", ".join(c["top_terms"]),
        "pct_standard_all": pct_genai(hls),
        "headline_count": c["headline_count"],
    })
diag_df = pd.DataFrame(diag_rows).sort_values(["period", "rank_in_week"])
print("\nDiagnostic — communities with genAI-vocabulary member terms (Nov–Dec 2022):")
if diag_df.empty:
    print("  (none — signal may be below Louvain resolution or min doc freq)")
else:
    print(diag_df.to_string(index=False))

# --- edge burst table (discovery window) ---
edge_rows = []
for week in sorted(graphs_by_week.keys(), key=week_ts):
    wts = week_ts(week)
    if wts < DISCOVERY_START or wts > DISCOVERY_END:
        continue
    graph = graphs_by_week[week]
    for a, b in graph.edges():
        eb = edge_burst(a, b, week)
        if eb <= 0:
            continue
        edge_rows.append({
            "period": week, "term_a": a, "term_b": b,
            "baseline_co": baseline_edge_presence(a, b),
            "current_co": edge_week_counts[(a, b) if a < b else (b, a)].get(week, 0),
            "edge_burst": eb,
        })
edge_df = pd.DataFrame(edge_rows).sort_values("edge_burst", ascending=False)
edge_df.to_parquet(EDGE_BURSTS_PATH, index=False)
print(f"\nWrote {EDGE_BURSTS_PATH.name} ({len(edge_df):,} edges)")
print("Top edge bursts (Nov–Dec 2022):")
print(edge_df.head(15).to_string(index=False))

# --- emergence timeline ---
top_tracks = (
    rank_df.loc[rank_df.period.map(week_ts).between(DISCOVERY_START, DISCOVERY_END)]
    .groupby("track_id")["emergence_score"].max()
    .sort_values(ascending=False).head(10).index.tolist()
)
fig = go.Figure()
for tid in top_tracks:
    sub = rank_df[rank_df.track_id == tid].sort_values("period")
    fig.add_trace(go.Scatter(
        x=sub["period"].map(week_ts), y=sub["emergence_score"],
        mode="lines+markers", name=f"track {tid}",
    ))
fig.add_vline(x=CHATGPT_LAUNCH, line_dash="dash", line_color="gray")
fig.update_layout(title="Top 10 emergence tracks (discovery window peak)", height=480, xaxis_title="week")
timeline_path = OUTPUT_DIR / "genai_graph_emergence_timeline.html"
fig.write_html(timeline_path)
print(f"Wrote {timeline_path.name}")

# --- network for peak discovery week ---
peak_week = (
    rank_df.loc[rank_df.period.map(week_ts).between(DISCOVERY_START, DISCOVERY_END)]
    .sort_values("emergence_score", ascending=False).iloc[0]["period"]
    if len(rank_df) else None
)
if peak_week and peak_week in graphs_by_week:
    g = graphs_by_week[peak_week]
    pos = nx.spring_layout(g, seed=RANDOM_SEED, k=0.5)
    edge_x, edge_y = [], []
    for a, b in g.edges():
        x0, y0 = pos[a]; x1, y1 = pos[b]
        edge_x += [x0, x1, None]; edge_y += [y0, y1, None]
    node_x = [pos[n][0] for n in g.nodes()]
    node_y = [pos[n][1] for n in g.nodes()]
    node_text = list(g.nodes())
    fig2 = go.Figure()
    fig2.add_trace(go.Scatter(x=edge_x, y=edge_y, mode="lines", line=dict(width=0.5, color="#888"), hoverinfo="none"))
    fig2.add_trace(go.Scatter(
        x=node_x, y=node_y, mode="markers+text", text=node_text,
        textposition="top center", marker=dict(size=8),
    ))
    fig2.update_layout(title=f"PPMI network — peak emergence week {peak_week}", showlegend=False, height=600)
    net_path = OUTPUT_DIR / "genai_graph_network_peak.html"
    fig2.write_html(net_path)
    print(f"Wrote {net_path.name}")

## Results interpretation

**Benchmark (post-hoc):** the graph pipeline surfaces genAI-vocabulary member terms in Nov–Dec 2022 communities (see diagnostic in §10), e.g. `chatgpt`/`openai` on **2022-12-13** (track 2838, rank #3). Representative-headline overlap with `GENAI_STANDARD` remains below the 20% target because Louvain merges the tiny genAI signal into large contemporaneous clusters (FTX, China protests, earnings).

**Edge bursts** in the discovery window are dominated by macro/event pairs; genAI edge bursts (`chatgpt–openai`, `chatgpt–promise`) exist but at much lower co-occurrence counts.

**Next tuning levers (if continuing):** higher Louvain resolution, 2-week rolling graphs, or seed-free sub-graph extraction around top edge-burst pairs — see [`alternative_plan_with_graph.md`](alternative_plan_with_graph.md) §19.

## Quick reload (no re-run)

Run §0 setup, then this cell, to inspect saved outputs.

In [ ]:
for path in [RANKINGS_PATH, VALIDATION_PATH, EDGE_BURSTS_PATH, COMMUNITIES_PATH]:
    if not path.exists():
        print(f"missing {path.name}")
        continue
    t = pd.read_parquet(path)
    print(f"\n{path.name} ({len(t):,} rows)")
    print(t.head(3).to_string(index=False))